# Main ingestion pipeline: PDF to Qdrant vector store

# Import

In [1]:
import torch
import pypdfium2 as pdfium
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor
import time
import json
import re
from collections import Counter
from unstructured.partition.md import partition_md
from unstructured.documents.elements import Title
from unstructured.chunking.title import chunk_by_title

c:\Users\calla\Documents\AI projects\chatbot\mas\code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading model

In [2]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"            # block any HF Hub call
os.environ["TRANSFORMERS_OFFLINE"] = "1"       # block transformers downloads

import torch
import pypdfium2 as pdfium
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor
import time

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
PDF_PATH = "1706.03762v7.pdf"
MODEL_PATH = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\chunk\LightOnOCR-2-1B"

# Load model and processor from local folder (no internet required)
model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="sdpa",
    local_files_only=True,
)
processor = LightOnOcrProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Model loaded from: {MODEL_PATH}")

You are using a model of type mistral3 to instantiate a model of type lighton_ocr. This is not supported for all configurations of models and can yield errors.
Loading weights: 100%|██████████| 532/532 [00:00<00:00, 1017.56it/s, Materializing param=model.vision_projection.patch_merger.merging_layer.weight]               


GPU: NVIDIA GeForce RTX 5090
Model loaded from: C:\Users\calla\Documents\AI projects\chatbot\mas\code\chunk\LightOnOCR-2-1B


# Inference

In [3]:
pdf = pdfium.PdfDocument(PDF_PATH)
images = [page.render(scale=1540/max(page.get_width(), page.get_height())).to_pil() for page in pdf]
print(f"{len(images)} pages")

15 pages


In [4]:
@torch.inference_mode()
def ocr_batch(images, max_new_tokens=1024):
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "image", "image": img}]} for img in images],
        add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", padding=True
    )
    inputs = {k: v.to("cuda", torch.bfloat16 if v.is_floating_point() else None) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    input_len = inputs["input_ids"].shape[1]
    return [processor.decode(seq[input_len:], skip_special_tokens=True) for seq in out]


In [5]:
BATCH_SIZE = 5
_ = ocr_batch(images[:BATCH_SIZE])

t0 = time.time()
results = []
for i in range(0, len(images), BATCH_SIZE):
    results.extend(ocr_batch(images[i:i + BATCH_SIZE]))

total = time.time() - t0
print(f"Total: {total:.1f}s pour {len(images)} pages ({total/len(images):.2f}s/page)")

Total: 83.3s pour 15 pages (5.55s/page)


In [6]:
import gc
import torch
import time
def clear_memory():
    print("\nNettoyage de la VRAM en cours...")

    # 1. Supprimer les variables qui pointent vers le modèle et le processeur
    try:
        del model
        del processor
    except NameError:
        pass # Au cas où ils seraient déjà supprimés

    # 2. Forcer le ramasse-miettes (Garbage Collector) de Python à faire le ménage
    gc.collect()

    # 3. Vider le cache CUDA pour rendre la mémoire au GPU
    torch.cuda.empty_cache()

    # Optionnel : réinitialiser les statistiques de mémoire CUDA
    torch.cuda.reset_peak_memory_stats()

    print("VRAM libérée avec succès ! Prêt pour le chargement des modèles d'embedding.")

In [7]:
clear_memory()


Nettoyage de la VRAM en cours...
VRAM libérée avec succès ! Prêt pour le chargement des modèles d'embedding.


# Chunking

In [8]:
# Simple unstructured chunking + RAG metadata (+ mini OCR-noise patch)



def _is_valid_title(text):
    # Ignore OCR titles like "5", "31", "14.2".
    return bool(re.search(r"[A-Za-z]", (text or "").strip()))


def _is_noise_chunk(text):
    tokens = re.findall(r"\b\w+\b", text or "")
    if not tokens:
        return True
    alpha = sum(any(c.isalpha() for c in t) for t in tokens)
    numeric = sum(t.isdigit() for t in tokens)
    return alpha < 3 or numeric > alpha * 3


def to_unstructured_hierarchical_chunks(markdown_batches, doc_id, batch_size, total_pages):
    elements, stack = [], []

    # Parse each OCR batch, then chunk once globally (no forced split at batch/page boundary).
    for batch_idx, md_text in enumerate(markdown_batches):
        page_start = batch_idx * batch_size + 1
        page_end = min((batch_idx + 1) * batch_size, total_pages)

        for el in partition_md(text=md_text):
            if isinstance(el, Title):
                title = el.text.strip()
                if not _is_valid_title(title):
                    continue
                depth = getattr(el.metadata, "category_depth", 0) or 0
                level = int(depth) + 1
                stack = stack[: max(0, level - 1)]
                stack.append(title)

            el._page_range = (page_start, page_end)  # coarse provenance from OCR batch
            el._section_path = list(stack)
            elements.append(el)

    chunks = chunk_by_title(
        elements,
        max_characters=1200,
        new_after_n_chars=1000,
        overlap=200,
        combine_text_under_n_chars=200,
    )

    out = []
    for i, ch in enumerate(chunks, 1):
        orig = getattr(getattr(ch, "metadata", None), "orig_elements", []) or []

        section_path = next(
            (getattr(oe, "_section_path", []) for oe in orig if getattr(oe, "_section_path", [])),
            [],
        )
        if not section_path and orig:
            section_path = getattr(orig[0], "_section_path", [])

        element_types = [type(oe).__name__ for oe in orig] or [type(ch).__name__]
        page_ranges = [getattr(oe, "_page_range", None) for oe in orig if getattr(oe, "_page_range", None)]
        page_start = min(r[0] for r in page_ranges) if page_ranges else None
        page_end = max(r[1] for r in page_ranges) if page_ranges else None
        text = ch.text

        out.append({
            "id": f"{doc_id}_c{i}",
            "text": text,
            "metadata": {
                "doc_id": doc_id,
                "chunk": i,
                "section_title": section_path[-1] if section_path else None,
                "section_path": section_path,
                "section_depth": len(section_path),
                "page_start": page_start,
                "page_end": page_end,
                "page_granularity": "batch_range",
                "chunk_type": Counter(element_types).most_common(1)[0][0],
                "element_types": sorted(set(element_types)),
                "has_table": "Table" in element_types,
                "has_list": any(t in {"ListItem", "List"} for t in element_types),
                "has_image": "Image" in element_types,
                "char_len": len(text),
                "word_count": len(text.split()),
                "approx_tokens": max(1, len(text) // 4),
            },
        })

    # Mini cleanup for OCR artifacts: drop noisy chunks, merge very short chunks into previous.
    cleaned = []
    for row in out:
        if _is_noise_chunk(row["text"]):
            continue
        if cleaned and len(row["text"]) < 260:
            prev = cleaned[-1]
            prev["text"] = f"{prev['text'].rstrip()}\n\n{row['text'].lstrip()}"
            pm, cm = prev["metadata"], row["metadata"]
            starts = [x for x in (pm.get("page_start"), cm.get("page_start")) if x is not None]
            ends = [x for x in (pm.get("page_end"), cm.get("page_end")) if x is not None]
            pm["page_start"] = min(starts) if starts else None
            pm["page_end"] = max(ends) if ends else None
            pm["element_types"] = sorted(set(pm.get("element_types", [])) | set(cm.get("element_types", [])))
            pm["has_table"] = pm.get("has_table", False) or cm.get("has_table", False)
            pm["has_list"] = pm.get("has_list", False) or cm.get("has_list", False)
            pm["has_image"] = pm.get("has_image", False) or cm.get("has_image", False)
        else:
            cleaned.append(row)
    out = cleaned

    total_chunks = len(out)
    for idx, row in enumerate(out, 1):
        m = row["metadata"]
        row["id"] = f"{doc_id}_c{idx}"
        m["chunk"] = idx
        m["char_len"] = len(row["text"])
        m["word_count"] = len(row["text"].split())
        m["approx_tokens"] = max(1, len(row["text"]) // 4)
        m["total_chunks"] = total_chunks
        m["position_ratio"] = round(idx / total_chunks, 3) if total_chunks else None
        m["prev_chunk_id"] = f"{doc_id}_c{idx - 1}" if idx > 1 else None
        m["next_chunk_id"] = f"{doc_id}_c{idx + 1}" if idx < total_chunks else None

    return out


final_chunks = to_unstructured_hierarchical_chunks(
    results,
    doc_id=PDF_PATH,
    batch_size=BATCH_SIZE,
    total_pages=len(images),
)

# Insertion qdrant

In [9]:
import json
import os

def save_chunks(chunks, filename="final_chunks.json"):
    """Saves the list of chunks to a JSON file."""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    print(f"Successfully saved {len(chunks)} chunks to {filename}")

def load_chunks(filename="final_chunks.json"):
    """Loads the list of chunks from a JSON file if it exists."""
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            chunks = json.load(f)
        print(f"Successfully loaded {len(chunks)} chunks from {filename}")
        return chunks
    print(f"No cache found at {filename}")
    return None

In [10]:
CACHE_FILE = f"{PDF_PATH}_chunks.json"

# 1. Try to load from cache first
final_chunks = load_chunks(CACHE_FILE)

# 2. If no cache, run the full pipeline
if final_chunks is None:
    print("Processing PDF (OCR + Chunking)...")

    # Run your existing chunking function
    final_chunks = to_unstructured_hierarchical_chunks(
        results,
        doc_id=PDF_PATH,
        batch_size=BATCH_SIZE,
        total_pages=len(images),
    )
    
    # Save to cache for next time
    save_chunks(final_chunks, CACHE_FILE)
else:
    print("Skipping OCR and chunking, using cached data.")

Successfully loaded 7 chunks from 1706.03762v7.pdf_chunks.json
Skipping OCR and chunking, using cached data.


## 

In [11]:
import subprocess
import requests
import time
import os

# check if qdrant responds on its default port
try:
    requests.get("http://localhost:6333")
    print("Qdrant is already running!")
except requests.exceptions.ConnectionError:
    print("Qdrant is not running. Attempting to start Docker container...")
    
    # Try to start the existing container first
    result = subprocess.run(["docker", "start", "qdrant_db"], capture_output=True, text=True)
    
    if result.returncode == 0:
         print("Existing Qdrant container started.")
    else:
        print("Container 'qdrant_db' not found or failed to start. Creating a new one...")
        # Fix the path formatting for Docker volume mount
        cwd = os.path.abspath(os.getcwd()).replace('\\', '/')
        volume_mount = f"{cwd}/qdrant_storage:/qdrant/storage"
        
        run_result = subprocess.run([
            "docker", "run", "-d", "-p", "6333:6333", "-p", "6334:6334",
            "-v", volume_mount, 
            "--name", "qdrant_db", "qdrant/qdrant"
        ], capture_output=True, text=True)
        
        if run_result.returncode != 0:
            print(f"ERROR starting container: {run_result.stderr}")
        else:
             print("New Qdrant container created and started.")
        
    # Wait a bit longer for initialization
    print("Waiting for Qdrant to initialize...")
    time.sleep(5)
    
    # Final check
    try:
        requests.get("http://localhost:6333")
        print("Qdrant successfully started and responding!")
    except requests.exceptions.ConnectionError:
        print("WARNING: Qdrant still not responding. Check Docker desktop or run 'docker logs qdrant_db' in your terminal.")

Qdrant is already running!


In [12]:
import qdrant_client
from qdrant_client.models import SparseVectorParams
from qdrant_client.models import Distance, VectorParams
from langchain_community.vectorstores import Qdrant
from langchain_community.embeddings import LlamaCppEmbeddings

collection_name = "pdf_chunks"
url = "http://localhost:6333"

qdrant = qdrant_client.QdrantClient(url=url)

In [13]:
VECTOR_DIMENSION = 4096
qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": VectorParams(size=VECTOR_DIMENSION, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

C:\Users\calla\AppData\Local\Temp\ipykernel_3556\3427014447.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

### Loading embeding

In [14]:
import os
import ctypes

# define cuda path
cuda_bin = r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0\bin\x64"

# 1 force path in os environment for c++ underlying loadlibrary calls
os.environ["PATH"] = cuda_bin + os.pathsep + os.environ.get("PATH", "")

# 2 force python dll directory
if os.path.exists(cuda_bin):
    os.add_dll_directory(cuda_bin)

# 3 test raw dll load to catch the exact missing dependency
dll_path = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\.venv\Lib\site-packages\llama_cpp\lib\llama.dll"
try:
    ctypes.CDLL(dll_path)
    print("raw dll loaded successfully")
except Exception as e:
    print(f"raw dll load failed: {e}")

# 4 load langchain model
from langchain_community.embeddings.llamacpp import LlamaCppEmbeddings

model_path = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\chunk\qwen_embedding\qwen-embed-8B.gguf"

# RTX 5090: larger token batch improves dense embedding throughput.
embeddings = LlamaCppEmbeddings(
    model_path=model_path,
    n_gpu_layers=-1,
    n_batch=1024,
    n_ctx=8192,
    verbose=False,
)

print("model loaded successfully")

raw dll loaded successfully


llama_context: n_ctx_per_seq (8192) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


model loaded successfully


### Loading sparse

In [15]:
from fastembed import SparseTextEmbedding

fastembed_model_id = "prithivida/Splade_PP_en_v1"

# path to the folder containing your downloaded hf model files
# make sure the onnx files and tokenizer configs are in here
local_splade_path = r"C:\Users\calla\Documents\AI projects\chatbot\mas\code\chunk\splade_local"

# load splade using fastembed
# batch size 512 is easy for an rtx 5090
# cuda execution provider forces it onto the gpu
sparse_embedder = SparseTextEmbedding(
    model_name=fastembed_model_id,
    cache_dir=local_splade_path,
    local_files_only=True,
    providers=["CUDAExecutionProvider"],
    batch_size=128
)

# quick test to ensure it works
texts = ["test document for sparse embedding"]
sparse_vectors = list(sparse_embedder.embed(texts))

# print first vector indices and values
print(sparse_vectors[0])

SparseEmbedding(values=array([0.12816659, 0.56002009, 0.26966175, 0.61802816, 2.28203988,
       0.06547009, 0.22684972, 0.8429935 , 1.09188557, 0.93930495,
       0.31223774, 0.57224894, 0.0927739 , 1.71002758, 0.9207651 ,
       0.2465439 , 1.98386526, 1.77601576, 1.12569249, 0.20433615,
       0.1845513 , 2.31647396]), indices=array([ 1012,  2000,  2005,  2793,  3231,  4007,  4289,  4638,  4667,
        5371,  5491,  5604,  5732,  6254,  6845,  6994,  7861,  8270,
        9742, 12827, 19701, 20288]))


In [16]:
clear_memory()


Nettoyage de la VRAM en cours...
VRAM libérée avec succès ! Prêt pour le chargement des modèles d'embedding.


## Add vectors

In [ ]:
import uuid
import numpy as np
from tqdm import tqdm
from qdrant_client.models import PointStruct

def batch_insert_to_qdrant(chunks, client, collection_name, dense_embedder, sparse_embedder, batch_size=128):
    """
    Embeds texts and inserts them into Qdrant with both dense and sparse vectors.
    Includes a sub-batching safety net for Llama.cpp to prevent decode crashes.
    """
    total_chunks = len(chunks)
    
    # Qdrant/FastEmbed run perfectly with large batch sizes (128)
    for i in tqdm(range(0, total_chunks, batch_size), desc="Inserting into Qdrant"):
        batch = chunks[i : i + batch_size]
        texts = [chunk["text"] for chunk in batch]
        
        # 1. Generate Dense Embeddings (one text at a time to avoid KV cache overflow)
        # llama_decode -1 = total tokens across all sequences exceeds n_ctx; embed 1 by 1 to be safe.
        dense_vectors = []
        for text in texts:
            dense_vectors.extend(dense_embedder.embed_documents([text]))
        
        # 2. Generate Sparse Embeddings (FastEmbed handles the full 128 list natively)
        sparse_vectors = list(sparse_embedder.embed(texts))
        
        points = []
        for j, chunk in enumerate(batch):
            # Qdrant requires UUIDs or integers. Hash the string ID into a UUID5.
            point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk["id"]))
            
            # Prepare the point payload (metadata + the raw text)
            payload = chunk.get("metadata", {})
            payload["text"] = chunk["text"]
            payload["original_id"] = chunk["id"]
            
            # Construct the point
            points.append(
                PointStruct(
                    id=point_id,
                    payload=payload,
                    vector={
                        "dense": dense_vectors[j],
                        "sparse": {
                            "indices": sparse_vectors[j].indices.tolist(),
                            "values": sparse_vectors[j].values.tolist(),
                        }
                    }
                )
            )
            
        # 3. Upload batch to Qdrant
        client.upload_points(
            collection_name=collection_name,
            points=points,
            wait=True
        )

# --- Execution ---
batch_insert_to_qdrant(
    chunks=final_chunks,
    client=qdrant,
    collection_name=collection_name,
    dense_embedder=embeddings,
    sparse_embedder=sparse_embedder,
    batch_size=20
)

Inserting into Qdrant:   0%|          | 0/1 [00:00<?, ?it/s]init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
Inserting into Qdrant: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s]


In [18]:
# --- Verification: check vectors inserted in Qdrant ---
collection_info = qdrant.get_collection(collection_name=collection_name)
total_points = collection_info.points_count

print(f"Collection : '{collection_name}'")
print(f"Nombre de vecteurs insérés : {total_points}")
print(f"Status : {collection_info.status}")

# Sample a few points to confirm payload + vectors are present
sample = qdrant.scroll(
    collection_name=collection_name,
    limit=3,
    with_payload=True,
    with_vectors=True
)

print(f"\n--- Exemple de {len(sample[0])} points ---")
for point in sample[0]:
    dense_dim = len(point.vector["dense"]) if point.vector and "dense" in point.vector else "N/A"
    sparse_nnz = len(point.vector["sparse"].indices) if point.vector and "sparse" in point.vector else "N/A"
    print(f"\nID       : {point.id}")
    print(f"Dense    : {dense_dim} dimensions")
    print(f"Sparse   : {sparse_nnz} valeurs non nulles")
    print(f"Payload  : {list(point.payload.keys())}")

Collection : 'pdf_chunks'
Nombre de vecteurs insérés : 7
Status : green

--- Exemple de 3 points ---

ID       : 0059fa50-f18b-588e-b01d-d3ab18e78248
Dense    : 4096 dimensions
Sparse   : 92 valeurs non nulles
Payload  : ['doc_id', 'chunk', 'section_title', 'section_path', 'section_depth', 'page_start', 'page_end', 'page_granularity', 'chunk_type', 'element_types', 'has_table', 'has_list', 'has_image', 'char_len', 'word_count', 'approx_tokens', 'total_chunks', 'position_ratio', 'prev_chunk_id', 'next_chunk_id', 'text', 'original_id']

ID       : 0260db80-56a7-5934-88a2-f0c6a0635459
Dense    : 4096 dimensions
Sparse   : 124 valeurs non nulles
Payload  : ['doc_id', 'chunk', 'section_title', 'section_path', 'section_depth', 'page_start', 'page_end', 'page_granularity', 'chunk_type', 'element_types', 'has_table', 'has_list', 'has_image', 'char_len', 'word_count', 'approx_tokens', 'total_chunks', 'position_ratio', 'prev_chunk_id', 'next_chunk_id', 'text', 'original_id']

ID       : 2b8e2669